# ANFIS

A lesson from the Python and AI School, by Dr. Behrooz Taheri

**ANFIS** (Jang, 1993) turns a Sugeno system into a five-layer network that learns from data.

- Layer 1: membership degrees (premise parameters)
- Layer 2: rule strength by product
- Layer 3: normalization
- Layer 4: rule output (consequent parameters)
- Layer 5: final sum

## Hybrid learning
In the forward pass, consequents are found by least squares; in the backward pass, premises are updated by gradient.

> There are $M^n$ rules, so many inputs bring the curse of dimensionality. Input selection or clustering helps.

## Layer by layer
With $x=3$ and $y=4$, Gaussian functions with $\sigma=2$ and centers A1=2, A2=6, B1=3, B2=7, and $f_1=0.5x+0.2y+1$ and $f_2=-0.3x+0.8y+2$:

| Layer | Calculation | Result |
|---|---|---|
| 1 | Membership | μA1=μB1=0.882, μA2=μB2=0.325 |
| 2 | Product | w1=0.779, w2=0.105 |
| 3 | Normalization | 0.881 and 0.119 |
| 4 | Rule output | 0.881×3.3=2.907 and 0.119×4.3=0.513 |
| 5 | Sum | 3.419 |

> [Membership functions evolving while training on the sinc function.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/anfis_mf.gif)

*Membership functions evolving while training on the sinc function.*

> [The target, the output before training, and after training.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/anfis_surf.png)

*The target, the output before training, and after training.*

> [The error curve and membership functions before and after training.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/anfis_curve.png)

*The error curve and membership functions before and after training.*

## Counting parameters
With M Gaussian functions for each of n inputs and first-order consequents:

| Part | Count |
|---|---|
| Rules | $M^n$ |
| Premise | $2Mn$ |
| Consequent | $(n+1)M^n$ |

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, (400, 2))
y = np.sin(X[:, 0]) * np.cos(X[:, 1])

centers = np.tile(np.array([-3.0, 0.0, 3.0]), (2, 1))
sigmas = np.full((2, 3), 1.5)

def design(X, c, s):
    m = np.exp(-(X[:, :, None] - c) ** 2 / (2 * s ** 2))
    w = (m[:, 0, :, None] * m[:, 1, None, :]).reshape(len(X), -1)
    nw = w / w.sum(1, keepdims=True)
    A = np.c_[X, np.ones(len(X))]
    return (nw[:, :, None] * A[:, None, :]).reshape(len(X), -1)

def solve(c, s):
    A = design(X, c, s)
    q = np.linalg.lstsq(A, y, rcond=None)[0]
    return q, float(np.sqrt(np.mean((A @ q - y) ** 2)))

q, e = solve(centers, sigmas)
print("start RMSE", round(e, 4))
lr = 0.1
for epoch in range(40):
    q, e = solve(centers, sigmas)
    g = np.zeros_like(centers)
    for i in np.ndindex(g.shape):
        c2 = centers.copy(); c2[i] += 1e-4
        g[i] = (solve(c2, sigmas)[1] - e) / 1e-4
    trial = centers - lr * g
    if solve(trial, sigmas)[1] < e: centers, lr = trial, lr * 1.2
    else: lr *= 0.5
print("final RMSE", round(solve(centers, sigmas)[1], 4))
print(centers.round(2))